In [ ]:
# Download data from Zenodo (skip if already downloaded)
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'scripts'))
from download_data import download_all
from config import DATA_ROOT

if not os.path.exists(os.path.join(DATA_ROOT, 'summary_stats.h5')):
    download_all()
else:
    print(f'Data already present at {DATA_ROOT}')

In [7]:
import os
import sys
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'scripts'))

import numpy as np
import pandas as pd
from config import DATA_ROOT

In [8]:
# Load pre-computed summary statistics (indexed by GW event name)
summary = pd.read_hdf(os.path.join(DATA_ROOT, 'summary_stats.h5'), key='summary')

# Load e_gw conversions
egw = pd.read_hdf(os.path.join(DATA_ROOT, 'egw_conversions.h5'), key='egw')

# Convert DataFrame rows to dict format for the table functions
table_stats = {}
for event, row in summary.iterrows():
    table_stats[event] = {
        'gw_event_name': event,
        'superevent': row.get('superevent', ''),
        'min_far': row.get('min_far', np.nan),
        'log10_bf_eas_qcas': row.get('log10_bf_eas_qcas', np.nan),
        'log10_bf_eas_qcp': row.get('log10_bf_eas_qcp', np.nan),
        'n_eff': row.get('n_eff', np.nan),
        'sample_efficiency': row.get('sample_efficiency', np.nan),
        'pe_algorithm': row.get('pe_algorithm', None),
        'mean_eccentricity': row.get('mean_eccentricity', np.nan),
        'hdi_eccentricity': np.array([row.get('eccentricity_hdi_lower', np.nan), row.get('eccentricity_hdi_upper', np.nan)]),
        'mean_chi_eff': row.get('mean_chi_eff', np.nan),
        'hdi_chi_eff': np.array([row.get('chi_eff_hdi_lower', np.nan), row.get('chi_eff_hdi_upper', np.nan)]),
        'mean_total_mass_src': row.get('mean_total_mass_src', np.nan),
        'hdi_total_mass_src': np.array([row.get('total_mass_src_hdi_lower', np.nan), row.get('total_mass_src_hdi_upper', np.nan)]),
        'mean_e_gw': np.nan,
        'hdi_e_gw': None,
    }

# Merge e_gw data (egw is indexed by superevent name)
for event, stats in table_stats.items():
    superevent = stats['superevent']
    if superevent in egw.index:
        stats['mean_e_gw'] = egw.loc[superevent, 'mean_e_gw']
        stats['hdi_e_gw'] = np.array([egw.loc[superevent, 'hdi_e_gw_lower'], egw.loc[superevent, 'hdi_e_gw_upper']])

print(f'Loaded table stats for {len(table_stats)} events')

Loaded table stats for 84 events


In [9]:
def color_for_bayes_factor(bf):
    if bf < -1.0: return '\\ColorOne'
    elif bf < -0.25: return '\\ColorTwo'
    elif bf < 0.25: return '\\ColorThree'
    elif bf < 0.75: return '\\ColorFour'
    elif bf < 1.25: return '\\ColorFive'
    elif bf < 1.75: return '\\ColorSix'
    elif bf < 2.25: return '\\ColorSeven'
    elif bf < 2.75: return '\\ColorEight'
    elif bf < 3.25: return '\\ColorNine'
    else: return '\\ColorTen'

def color_for_sample_efficiency(eff):
    eff_percent = eff * 100
    if eff_percent < 0.5: return '\\ColorOne'
    elif eff_percent < 1.0: return '\\ColorTwo'
    elif eff_percent < 5.0: return '\\ColorFour'
    elif eff_percent < 10.0: return '\\ColorFive'
    elif eff_percent < 20.0: return '\\ColorSix'
    elif eff_percent < 30.0: return '\\ColorSeven'
    elif eff_percent < 40.0: return '\\ColorNine'
    else: return '\\ColorTen'

def color_for_ess(ess):
    if ess < 1000: return '\\ColorOne'
    elif ess < 5000: return '\\ColorThree'
    else: return '\\ColorTen'

def float_to_latex_scientific(val, precision=1):
    try:
        val = float(val)
    except (TypeError, ValueError):
        return '$-$'
    if np.isnan(val): return '$\\mathrm{NaN}$'
    if np.isinf(val): return '$\\infty$' if val > 0 else '$-\\infty$'
    s = f'{val:.{precision}e}'
    if 'e' not in s: return f'${val:.{precision}f}$'
    base_str, exp_str = s.split('e')
    base, exp = float(base_str), int(exp_str)
    if exp == 0: return f'${base:.{precision}f}$'
    elif abs(exp) == 1:
        return f'${base * (10 ** exp):.{precision}f}$'
    else:
        return f'${base:.{precision}f} \\times 10^{{{exp}}}$'

def table_stat_row_to_latex_strs(table_stat_row):
    ret_dict = {}
    for key in ['eccentricity', 'e_gw', 'chi_eff', 'total_mass_src']:
        if f'mean_{key}' not in table_stat_row or table_stat_row[f'hdi_{key}'] is None or table_stat_row[f'mean_{key}'] is None:
            ret_dict[key] = '-'
        elif np.any(np.isnan(table_stat_row[f'hdi_{key}'])) or np.isnan(table_stat_row[f'mean_{key}']):
            ret_dict[key] = '-'
        else:
            decimals = 1 if key == 'total_mass_src' else 2
            ci = table_stat_row[f'hdi_{key}'] - table_stat_row[f'mean_{key}']
            ret_dict[key] = f'${table_stat_row[f"mean_{key}"]:.{decimals}f}_{{{ci[0]:.{decimals}f}}}^{{+{ci[1]:.{decimals}f}}}$'

    if table_stat_row['min_far'] <= 1e-5:
        ret_dict['min_far'] = '$< 1.0 \\times 10^{-5}$'
    else:
        ret_dict['min_far'] = float_to_latex_scientific(table_stat_row['min_far'], precision=1)

    ret_dict['gw_event_name'] = str(table_stat_row['gw_event_name']).replace('_', '\\_')

    if table_stat_row.get('pe_algorithm') == 'dingo':
        n_eff = table_stat_row['n_eff']
        n_eff_color = color_for_ess(n_eff) if not np.isnan(n_eff) else ''
        ret_dict['n_eff'] = f'{n_eff_color}${n_eff:.0f}$' if not np.isnan(n_eff) else '-'
        se = table_stat_row['sample_efficiency']
        if se is not None and not np.isnan(se):
            ret_dict['sample_efficiency'] = color_for_sample_efficiency(se) + f'${se*100:.1f}\\%$'
        else:
            ret_dict['sample_efficiency'] = '-'
    else:
        ret_dict['n_eff'] = '-'
        ret_dict['sample_efficiency'] = '-'

    for key in ['log10_bf_eas_qcas', 'log10_bf_eas_qcp']:
        if table_stat_row[key] is None or np.isnan(table_stat_row[key]):
            ret_dict[key] = '-'
        else:
            ret_dict[key] = color_for_bayes_factor(table_stat_row[key]) + f'${table_stat_row[key]:.2f}$'

    log10_prior_odds = np.log10(0.023)
    odds_val = log10_prior_odds + table_stat_row['log10_bf_eas_qcas']
    ret_dict['odds'] = color_for_bayes_factor(odds_val) + float_to_latex_scientific(odds_val)

    return ret_dict

In [10]:
def table_1_stats(table_stats):
    filtered_stats = {k: v for k, v in table_stats.items()
                     if not np.isnan(v['log10_bf_eas_qcas']) and v['log10_bf_eas_qcas'] > 0.2}
    table_stats_sorted = dict(
        sorted(filtered_stats.items(), key=lambda item: item[1]['gw_event_name'])
    )
    table_txt = """
\\begin{center}
\t\\setlength{\\tabcolsep}{6pt}
\t\\renewcommand{\\arraystretch}{1.5}
    \\begin{tabular}{lCCCCCCCCC}
        \\toprule
        Event name & $\\text{FAR}_{\\text{min}}$ ($\\text{yr}^{-1}$) & $\\log_{10} \\mathcal{B}_{\\text{EAS/QCAS}}$ & $\\log_{10} \\mathcal{B}_{\\text{EAS/QCP}}$ & $\\log_{10} \\mathcal{O}_{\\text{EAS/QCAS}}$ & $e_{10\\text{Hz}}$ & $e_{\\text{gw, 10Hz}}$ & $\\chi_{\\text{eff}}$ & $\\text{M}_{\\text{src, total}} [ M_\\odot ]$ \\\\\\\\
        \\midrule
"""
    for event, stats in table_stats_sorted.items():
        latex_row = table_stat_row_to_latex_strs(stats)
        table_txt += '\t\t\t' + ' & '.join([
            latex_row['gw_event_name'], latex_row['min_far'],
            latex_row['log10_bf_eas_qcas'], latex_row['log10_bf_eas_qcp'],
            latex_row['odds'], latex_row['eccentricity'],
            latex_row['e_gw'], latex_row['chi_eff'],
            latex_row['total_mass_src']
        ]) + ' \\\\\\\\ \n'
    table_txt += """
        \\bottomrule
    \\end{tabular}
\\end{center}
"""
    return table_txt

def table_2(table_stats):
    table_stats_sorted = dict(
        sorted(table_stats.items(), key=lambda item: item[1]['gw_event_name'])
    )
    items = list(table_stats_sorted.items())
    mid = len(items) // 2
    first_half = dict(items[:mid])
    second_half = dict(items[mid:])

    table_txt = """
\\begin{center}
\t\\rowcolors{2}{gray!25}{white}
\t\\setlength{\\tabcolsep}{2pt}
    \\begin{minipage}{0.48\\textwidth}
        \\begin{footnotesize}
            \\begin{tabular}{lCCCCCCC}
                Event name & $\\text{FAR}_{\\text{min}}$ ($\\text{yr}^{-1}$) & $e_{10\\text{Hz}}$ & $\\log_{10} \\mathcal{B}$ & $n_{\\text{eff}}$ & $n_{\\text{eff}}/N \\times 100$ \\\\\\\\
                \\hline
"""
    for event, stats in first_half.items():
        lr = table_stat_row_to_latex_strs(stats)
        table_txt += '\t\t\t\t' + ' & '.join([lr['gw_event_name'], lr['min_far'], lr['eccentricity'], lr['log10_bf_eas_qcas'], lr['n_eff'], lr['sample_efficiency']]) + ' \\\\\\\\ \n'
    table_txt += """
            \\end{tabular}
        \\end{footnotesize}
    \\end{minipage}
    \\hspace{0.0\\textwidth}
    \\begin{minipage}{0.48\\textwidth}
        \\begin{footnotesize}
            \\begin{tabular}{lCCCCC}
                Event name & $\\text{FAR}_{\\text{min}}$ ($\\text{yr}^{-1}$) & $e_{10\\text{Hz}}$ & $\\log_{10} \\mathcal{B}$ & $n_{\\text{eff}}$ & $n_{\\text{eff}}/N \\times 100$ \\\\\\\\
                \\hline
"""
    for event, stats in second_half.items():
        lr = table_stat_row_to_latex_strs(stats)
        table_txt += '\t\t\t\t' + ' & '.join([lr['gw_event_name'], lr['min_far'], lr['eccentricity'], lr['log10_bf_eas_qcas'], lr['n_eff'], lr['sample_efficiency']]) + ' \\\\\\\\ \n'
    table_txt += """
            \\end{tabular}
        \\end{footnotesize}
    \\end{minipage}
\\end{center}
"""
    return table_txt

In [11]:
print(table_1_stats(table_stats))


\begin{center}
	\setlength{\tabcolsep}{6pt}
	\renewcommand{\arraystretch}{1.5}
    \begin{tabular}{lCCCCCCCCC}
        \toprule
        Event name & $\text{FAR}_{\text{min}}$ ($\text{yr}^{-1}$) & $\log_{10} \mathcal{B}_{\text{EAS/QCAS}}$ & $\log_{10} \mathcal{B}_{\text{EAS/QCP}}$ & $\log_{10} \mathcal{O}_{\text{EAS/QCAS}}$ & $e_{10\text{Hz}}$ & $e_{\text{gw, 10Hz}}$ & $\chi_{\text{eff}}$ & $\text{M}_{\text{src, total}} [ M_\odot ]$ \\\\
        \midrule
			GW230706\_104333 & $0.2$ & \ColorFour$0.51$ & \ColorFour$0.40$ & \ColorOne$-1.1$ & $0.31_{-0.19}^{+0.18}$ & $0.30_{-0.17}^{+0.19}$ & $0.11_{-0.15}^{+0.15}$ & $27.3_{-3.3}^{+3.3}$ \\\\ 
			GW230709\_122727 & $7.1 \times 10^{-2}$ & \ColorFour$0.27$ & \ColorThree$0.19$ & \ColorOne$-1.4$ & $0.32_{-0.27}^{+0.24}$ & $0.33_{-0.28}^{+0.23}$ & $0.05_{-0.30}^{+0.32}$ & $72.2_{-14.8}^{+15.4}$ \\\\ 
			GW230820\_212515 & $0.2$ & \ColorFour$0.41$ & \ColorFour$0.50$ & \ColorOne$-1.2$ & $0.36_{-0.19}^{+0.17}$ & $0.34_{-0.17}^{+0.16}$ & $0.10_{-0.3

In [12]:
print(table_2(table_stats))


\begin{center}
	\rowcolors{2}{gray!25}{white}
	\setlength{\tabcolsep}{2pt}
    \begin{minipage}{0.48\textwidth}
        \begin{footnotesize}
            \begin{tabular}{lCCCCCCC}
                Event name & $\text{FAR}_{\text{min}}$ ($\text{yr}^{-1}$) & $e_{10\text{Hz}}$ & $\log_{10} \mathcal{B}$ & $n_{\text{eff}}$ & $n_{\text{eff}}/N \times 100$ \\\\
                \hline
				GW230601\_224134 & $< 1.0 \times 10^{-5}$ & $0.20_{-0.20}^{+0.19}$ & \ColorTwo$-0.32$ & \ColorTen$46091$ & \ColorSix$18.4\%$ \\\\ 
				GW230605\_065343 & $< 1.0 \times 10^{-5}$ & $0.13_{-0.13}^{+0.12}$ & \ColorTwo$-0.48$ & \ColorThree$1311$ & \ColorOne$0.1\%$ \\\\ 
				GW230606\_004305 & $1.3 \times 10^{-3}$ & $0.14_{-0.14}^{+0.16}$ & \ColorTwo$-0.59$ & \ColorTen$43038$ & \ColorFour$2.0\%$ \\\\ 
				GW230608\_205047 & $1.2 \times 10^{-3}$ & $0.18_{-0.18}^{+0.17}$ & \ColorTwo$-0.25$ & - & - \\\\ 
				GW230609\_064958 & $1.4 \times 10^{-4}$ & $0.18_{-0.18}^{+0.17}$ & \ColorTwo$-0.39$ & \ColorTen$54485$ & \ColorS